# LlaMa-3.2-3B-Instract — 移行版
環境依存の認証・パス・インストールを調整した実験ログです。全セル一括実行用ではありません。
元のセル順・実験条件は維持しています。必要なセクションと前提変数を選んで実行してください。
Llama/GemmaのPPL・LoRAはCUDA GPUを想定。Gemma原本にはモデル削除後の参照、Llama原本には未定義変数の参照があります。
元の出力はDrive原本に保存。現在のコードで同じ結果を再現できることは未検証です。
保存先・Colab接続・既知の制約は `docs/notebook_execution.md` を参照。


In [ ]:
from pathlib import Path
import os, sys
# On a remote Colab kernel, first upload/clone this repository into /content/RMT_utils.
candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/RMT_utils')]
ROOT = next((p for p in candidates if (p / 'notebook_runtime.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Upload or clone RMT_utils on this kernel, then rerun this cell.')
sys.path.insert(0, str(ROOT))
from notebook_runtime import setup_auth, data_dir, output_dir
setup_auth()
os.environ.setdefault('WANDB_MODE', 'offline')
print('Repository:', ROOT)
print('Data:', data_dir())
print('Outputs:', output_dir())


https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct


# モデルの読み込み

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM
# Colab-specific setup is handled by notebook_runtime.
# 1. シークレットからトークンを読み込んで環境変数にセット
setup_auth()

# 本家 Llama-3.2-3B のモデルIDを指定
model_id = "meta-llama/Llama-3.2-3B-Instruct"

print(f"{model_id} を読み込んでいます...")
# 3Bモデルはfp16で約6.5GBのため、無料版ColabのRAM空き容量に安全に収まります
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="cpu" # 念のため一度CPUに展開
)

# 2. ターゲット層の抽出
# 例として、中間層（Layer 14）の Self-Attention における q_proj を抽出します
layer_idx = 14
target_layer = model.model.layers[layer_idx].self_attn.q_proj

# 重みテンソルを取得し、NumPy配列に変換
W = target_layer.weight.detach().cpu().numpy().astype(np.float32)
print(f"\n[成功] 重み行列を抽出しました (Shape: {W.shape})")
print(f"  - 出力次元 (out_features): {W.shape[0]}")
print(f"  - 入力次元 (in_features): {W.shape[1]}")

In [ ]:
# 3. 特異値分解 (SVD) とランダム行列理論(RMT)に基づく固有値計算
print("\n特異値分解（SVD）を計算しています...")
_, S, _ = np.linalg.svd(W, full_matrices=False)

# 相関行列 X = (1/N) * W^T * W の固有値に変換するため、特異値を2乗してサンプリング次元（行数）で割る
# これにより西川氏の論文およびMarchenko-Pastur（MP）分布の理論曲線と完全に一致します
eigenvalues = (S ** 2) / W.shape[0]

print(f"計算された固有値の総数: {len(eigenvalues)}")
print(f"最大固有値 ($\lambda_{{max}}$): {np.max(eigenvalues):.6f}")
print(f"最小固有値 ($\lambda_{{min}}$): {np.min(eigenvalues):.6f}")

# 4. ESD (経験的スペクトル密度) のプロット
plt.figure(figsize=(10, 6))

# 確率密度関数として正規化（density=True）
# plt.hist(eigenvalues, bins=100, density=True, alpha=0.6, color='royalblue', edgecolor='black', label="Llama-3.2 ESD")
plt.hist(eigenvalues, bins=100, density=False, alpha=0.6, color='royalblue', edgecolor='black', label="Llama-3.2 ESD")

plt.title(f"Empirical Spectral Density (ESD) - Llama-3.2-3B Layer {layer_idx} q_proj", fontsize=13)
plt.xlabel("Eigenvalue $\lambda$", fontsize=11)
plt.ylabel("Probability Density $P(\lambda)$", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)

plt.show()

In [ ]:
# 4. ESD (経験的スペクトル密度) のプロット
plt.figure(figsize=(10, 6))

# 確率密度関数として正規化（density=True）
# plt.hist(eigenvalues, bins=100, density=True, alpha=0.6, color='royalblue', edgecolor='black', label="Llama-3.2 ESD")
plt.hist(S, bins=100, density=False, alpha=0.6, color='royalblue', edgecolor='black', label="Llama-3.2 ESD")

plt.title(f"Empirical Spectral Density (ESD) - Llama-3.2-3B Layer {layer_idx} q_proj", fontsize=13)
plt.xlabel("Singular $\lambda$", fontsize=11)
plt.ylabel("Probability Density $P(\lambda)$", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)

plt.show()

# Dyson Equalizerの適用

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def dyson_equalizer_algorithm1(Y):
    """
    Landa & Kluger (2024) - Algorithm 1: The Dyson Equalizer
    論文の数式と記法に完全に対応させた実装。

    Input:
        Y: Data matrix (m x n), m <= n
    Returns:
        Y_hat: Normalized data matrix
        x_hat: Row scaling vector
        y_hat: Column scaling vector
    """
    m, n = Y.shape
    if m > n:
        raise ValueError("Input matrix Y must have m <= n. Transpose Y if necessary.")

    # 1: Compute the SVD of Y
    # U: m x m, sigma: m, V_h: n x n
    U, sigma, V_h = np.linalg.svd(Y, full_matrices=True)
    V = V_h.T  # V \in R^{n x n} (右特異ベクトルを列に持つ行列)

    # 2: Set eta as the median singular value of Y
    eta = np.median(sigma)

    # 3: Compute the vectors g_hat^(1) and g_hat^(2)
    # 論文 (3) 式の計算（行列演算で高速化）
    term1 = eta / (sigma**2 + eta**2)
    term2 = term1 - (1 / eta)

    # U は m x m, sigma は要素数 m
    g1_hat = (U**2) @ term1

    # V は n x n. sum は k=1 から m までなので V の最初の m 列を使用
    g2_hat = (1 / eta) + (V[:, :m]**2) @ term2

    # 4: Compute the vectors x_hat and y_hat
    # L1ノルム ||g_hat^(1)||_1 と ||g_hat^(2)||_1 の計算
    g1_norm1 = np.sum(np.abs(g1_hat))
    g2_norm1 = np.sum(np.abs(g2_hat))

    # 論文 (4) 式の計算
    x_hat = (1 / np.sqrt(m - eta * g1_norm1)) * ((1 / g1_hat) - eta)
    y_hat = (1 / np.sqrt(n - eta * g2_norm1)) * ((1 / g2_hat) - eta)

    # 数値的安定性のための安全策（負値の平方根エラー回避）
    x_hat = np.maximum(1e-12, x_hat)
    y_hat = np.maximum(1e-12, y_hat)

    # 5: Form the normalized data matrix Y_hat
    # Y_hat = (D_{x_hat})^{-1/2} Y (D_{y_hat})^{-1/2}
    Y_hat = Y / (np.sqrt(x_hat[:, None]) * np.sqrt(y_hat[None, :]))

    return Y_hat, x_hat, y_hat

# ==========================================
# 動作検証用シミュレーション
# ==========================================
def run_simulation():
    print("もとの重みのSVDを計算")
    m, n = W.shape[0], W.shape[1]
    gamma = m / n
    _, S, _ = np.linalg.svd(W, full_matrices=False)

    # ==========================================
    # Algorithm 1 の適用
    # ==========================================
    print("Algorithm 1 (Dyson Equalizer) を適用中...")
    W_hat, x_hat, y_hat = dyson_equalizer_algorithm1(W)
    print("DE完了")

    # 補正後のESD計算 (特異値 S_hat)
    W_hat_scaled = W_hat / np.sqrt(n) # 実装上の修正　スケールを揃える
    _, S_hat, _ = np.linalg.svd(W_hat_scaled, full_matrices=False)

    # ---------------------------------------------------------
    # 【修正】特異値空間における理論曲線の計算 (MP分布の変数変換)
    # ---------------------------------------------------------
    lambda_plus = (1 + np.sqrt(gamma))**2
    lambda_minus = (1 - np.sqrt(gamma))**2

    # 横軸は特異値の軸 (sigma) として定義する
    sigma_max = np.sqrt(lambda_plus)
    sigma_min = np.sqrt(lambda_minus)
    x_ax = np.linspace(max(0, sigma_min - 0.5), sigma_max + 0.5, 1000)

    # 特異値の2乗 (lambda) がMPのサポート内にあるか判定
    valid_mask = (x_ax >= sigma_min) & (x_ax <= sigma_max)
    rho_singular = np.zeros_like(x_ax)

    # ヤコビアン 2*sigma を掛け合わせた特異値空間の理論式
    sigma_val = x_ax[valid_mask]
    lam_val = sigma_val ** 2
    rho_mp_converted = np.sqrt((lambda_plus - lam_val) * (lam_val - lambda_minus)) / (2 * np.pi * gamma * lam_val)
    rho_singular[valid_mask] = 2 * sigma_val * rho_mp_converted

    # ---------------------------------------------------------
    # プロット
    # ---------------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    bins = 100

    # 元の特異値分布
    axes[0].hist(S, bins=bins, density=True, color='salmon', alpha=0.8, label='Empirical $S$')
    axes[0].plot(x_ax, rho_singular, 'k--', lw=2, label='Theoretical Singular Law')
    axes[0].set_title('Original Singular Value Density', fontsize=12)
    axes[0].set_xlabel('Singular Value $\sigma$')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # DE適用後の特異値分布
    axes[1].hist(S_hat, bins=bins, density=True, color='dodgerblue', alpha=0.8, label='Empirical $\hat{S}$')
    axes[1].plot(x_ax, rho_singular, 'k--', lw=2, label='Theoretical Singular Law')
    axes[1].set_title('Dyson Equalizer Corrected Density', fontsize=12)
    axes[1].set_xlabel('Singular Value $\sigma$')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    run_simulation()

In [ ]:
# Environment setup is managed outside this experiment cell.
import sys
sys.path.insert(0, str(ROOT))

import funcs1

In [ ]:
# 4分くらい
# result = funcs1.get_esd_metrics(model)

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
# Colab-specific setup is handled by notebook_runtime.
# 1. Google Drive のマウント
# Mount Drive explicitly using the Colab extension if needed.

# 保存先のディレクトリを作成（ご自身の環境に合わせて変更してください）
save_dir = str(data_dir())
os.makedirs(save_dir, exist_ok=True)

filename = "llama-3.2-3B-instruct_matrics.pkl"
filepath = os.path.join(save_dir, filename)

if os.path.exists(filepath):
    result = pd.read_pickle(filepath)
    print(f"✅ データの読み込みが完了しました: {filepath}")
    print(f"行数（層の数）: {len(result)}")
else:
    print(f"❌ ファイルが見つかりません: {filepath}")

In [ ]:
result['name']

In [ ]:
# 探したい層の名前を指定（LLaMAなどのモデル構造に合わせる）
target_layer_name = "lm_head"

# results['name'] の中から、該当する層のインデックスを探す
try:
    target_index = result['name'].index(target_layer_name)

    # 同じインデックスを使って alpha と alphahat を取得
    layer_alpha = result['alpha'][target_index]
    layer_alphahat = result['alphahat'][target_index]

    print(f"Layer: {target_layer_name}")
    print(f"Alpha: {layer_alpha:.4f}")
    print(f"AlphaHat: {layer_alphahat:.4f}")

except ValueError:
    print(f"エラー: {target_layer_name} が results の中に見つかりませんでした。")

In [ ]:
layer_idx = 2
target_layer = model.model.layers[layer_idx].self_attn.q_proj
# target_layer = model.lm_head

# 重みテンソルを取得し、NumPy配列に変換
W = target_layer.weight.detach().cpu().numpy().astype(np.float32)
print(f"\n[成功] 重み行列を抽出しました (Shape: {W.shape})")
print(f"  - 出力次元 (out_features): {W.shape[0]}")
print(f"  - 入力次元 (in_features): {W.shape[1]}")

# テンソルがPyTorchの場合はNumPyに変換
if hasattr(W, 'detach'):
    W_np = W.detach().cpu().numpy()
else:
    W_np = W

m, n = W_np.shape
gamma = m / n  # LLaMAのq_projなどは 3072x3072 なので gamma=1.0

# 1. 経験的相関行列と固有値の計算
print("固有値を計算中...")
X = W_np @ W_np.T / n
evals = np.linalg.eigvalsh(X)

# 3. 特異値分解 (SVD) とランダム行列理論(RMT)に基づく固有値計算
print("\n特異値分解（SVD）を計算しています...")
# _, S, _ = np.linalg.svd(W, full_matrices=False)

gamma = W.shape[1] / W.shape[0]
m, n = W.shape[1], W.shape[0]
# result_orig = funcs1.bema_algorithm1_from_data(W, alpha=0.2, beta=0.1)

print("BEMA Algorithm 1")
print(f"sigma^2_hat = {result_orig['sigma2_hat']:.4f}")
print(f"K_hat        = {result_orig['K_hat']}")
print(f"threshold    = {result_orig['threshold']:.4f}")

sigma2_bema = result_orig['sigma2_hat']
K_bema = result_orig['K_hat']
threshold_bema = result_orig['threshold']
lambda_plus_bema = sigma2_bema * (1 + np.sqrt(gamma))**2
lambda_minus = sigma2_bema * (1 - np.sqrt(gamma))**2
x = np.linspace(lambda_minus, lambda_plus_bema, 500)
rho_mp_bema = np.sqrt((lambda_plus_bema - x) * (x - lambda_minus)) / (2 * np.pi * gamma * x * sigma2_bema)

# 3. プロットの作成
plt.figure(figsize=(14, 6))

# --- 右図：スパイクを含む全体像 ---
plt.hist(evals, bins=150, density=True, color='salmon', alpha=0.7, label='All Eigenvalues')
plt.plot(x, rho_mp_bema, 'b-', lw=2, label='Theoretical MP')
plt.axvline(threshold_bema, color='red', linestyle='--', lw=2, label='Threshold (Signal Start)')

plt.xlim(0, np.max(evals) * 1.05)
# スパイクの密度は非常に低いため、y軸を対数スケールにして見やすくする
plt.yscale('log')
plt.title(f'Full Range: {result_orig["K_hat"]} Spikes (Heavy-Tail)', fontsize=14)
plt.xlabel('Eigenvalue $\lambda$', fontsize=12)
plt.legend()

plt.show()

In [ ]:
layer_idx = 1
target_layer = model.model.layers[layer_idx].self_attn.q_proj
# target_layer = model.lm_head

# 重みテンソルを取得し、NumPy配列に変換
W = target_layer.weight.detach().cpu().numpy().astype(np.float32)
print(f"\n[成功] 重み行列を抽出しました (Shape: {W.shape})")
print(f"  - 出力次元 (out_features): {W.shape[0]}")
print(f"  - 入力次元 (in_features): {W.shape[1]}")

m, n = W.shape

if m > n:
  m, n = n ,m

  W = W.T

gamma = m / n  # LLaMAのq_projなどは 3072x3072 なので gamma=1.0

W = W * np.sqrt(n)

# 3. 特異値分解 (SVD) とランダム行列理論(RMT)に基づく固有値計算
print("\n特異値分解（SVD）を計算しています...")
_, S, _ = np.linalg.svd(W, full_matrices=False)
evals = (S ** 2) / n # スケールした相関行列の固有値

result_orig = funcs1.bema_algorithm1_from_data(W, alpha=0.2, beta=0.1)

print("BEMA Algorithm 1")
print(f"sigma^2_hat = {result_orig['sigma2_hat']:.4f}")
print(f"K_hat        = {result_orig['K_hat']}")
print(f"threshold    = {result_orig['threshold']:.4f}")

sigma2_bema = result_orig['sigma2_hat']
K_bema = result_orig['K_hat']
threshold_bema = result_orig['threshold']
lambda_plus_bema = sigma2_bema * (1 + np.sqrt(gamma))**2
lambda_minus = sigma2_bema * (1 - np.sqrt(gamma))**2
x = np.linspace(lambda_minus, lambda_plus_bema, 500)
rho_mp_bema = np.sqrt((lambda_plus_bema - x) * (x - lambda_minus)) / (2 * np.pi * gamma * x * sigma2_bema)

# 3. プロットの作成
plt.figure(figsize=(14, 6))

plt.hist(evals, bins=150, density=True, color='salmon', alpha=0.7, label='All Eigenvalues')
plt.plot(x, rho_mp_bema, 'b-', lw=2, label='Theoretical MP')
plt.axvline(threshold_bema, color='red', linestyle='--', lw=2, label='Threshold (Signal Start)')

plt.xlim(0, np.max(evals) * 1.05)
# スパイクの密度は非常に低いため、y軸を対数スケールにして見やすくする
plt.yscale('log')
plt.xlabel('Eigenvalue $\lambda$', fontsize=12)
plt.legend()

plt.show()

In [ ]:
sigma2_bema = funcs1.apply_bema(evals, gamma, m, alpha=0.2)

In [ ]:
print(sigma2_bema)